## 1. Configuração de credenciais
Configura o token de autenticação do Hugging Face a partir dos *secrets* do Google Colab e definindo-o como variável de ambiente. Esse token é necessário para baixar modelos e datasets que exigem autenticação no Hugging Face.

In [ ]:
import os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')

## 2. Instalação da biblioteca Unsloth
Uma biblioteca que otimiza o carregamento e a inferência de LLMs, tornando-os mais rápidos e eficientes em termos de memória.

In [ ]:
!pip install unsloth

## 3. Carregamento do modelo de linguagem
Carrega o modelo `Qwen/Qwen3.5-4B` utilizando o `FastLanguageModel` do Unsloth sem quantização em 4 bits (`load_in_4bit = False`) e prepara o modelo para o modo de inferência.

In [ ]:
import torch
from unsloth import FastLanguageModel

model_id = "Qwen/Qwen3.5-4B"
max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_id,
    max_seq_length = max_seq_length,
    dtype = torch.bfloat16,
    load_in_4bit = False,
)

FastLanguageModel.for_inference(model)

## 4. Carregamento do dataset de avaliação
Carrega o dataset `codeparrot/apps` (conjunto de problemas de programação competitiva) do Hugging Face.

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files="hf://datasets/codeparrot/apps/test.jsonl",
    split="train"
)

## 5. Inspeção do dataset
Exibe informações sobre o dataset carregado para entender o formato dos dados antes de utilizá-los.

In [ ]:
print(type(dataset[0]))
print(dataset.column_names)
print(dataset[0])

## 6. Definição dos prompts
Define o *system prompt* e o template `prompt_base`, usado para inserir o enunciado de cada problema (`question`) na mensagem enviada ao modelo.

In [ ]:
system_prompt = (
    "Act as an expert competitive programmer. Solve the following problem using Python 3. "
    "Output ONLY the raw, executable Python code. "
    "Do not include any explanations, greetings, comments, or markdown formatting (do NOT use ```python). "
    "Your entire response must be valid code ready to be submitted to an online judge."
)

prompt_base = "Problem description:\n{question}"

## 7. Geração das respostas e avaliação do modelo
Esta é a célula principal do experimento. Ela realiza as seguintes etapas:
- Cria um diretório no drive onde os resultados serão salvos, com backups periódicos;
- Inicializa um contador para limitar a quantidade de questões processadas por nível de dificuldade;
- Itera sobre todas as questões do dataset, monta o prompt e a pergunta, e envia ao modelo;
- Gera a resposta do modelo;
- Trata eventuais exceções durante a geração, registrando o erro no lugar da resposta;
- Grava cada resultado (prompt, resposta, id da questão e dificuldade) em um arquivo CSV local (`resultados.csv`);
- A cada 10 questões processadas, salva um backup do CSV no Google Drive.

In [ ]:
import csv
import time
import os
import shutil
import traceback
from google.colab import drive

drive.mount('/content/drive')
drive_dir = '/content/drive/MyDrive/resultados_eval_llm'
os.makedirs(drive_dir, exist_ok=True)
drive_path = os.path.join(drive_dir, f"{model_id.split('/')[1]}_eval.csv")

qnt = {
    "competition": 0,
    "interview": 0,
    "introductory": 0
    }

model.generation_config.max_length = None

with open('resultados.csv', 'w', newline='', encoding='utf-8') as f:
  writer = csv.DictWriter(f, fieldnames=['prompt', 'response', 'question_id', 'difficulty', 'time'])
  writer.writeheader()

  for i in range(len(dataset)):
    questao = dataset[i]
    if qnt[questao.get("difficulty")] <= 333:
      qnt[questao.get("difficulty")] += 1

      pergunta = questao.get("question")
      prompt = prompt_base.format(question=pergunta)

      try:

        messages = [
          {"role": "system", "content": [{"type": "text", "text": system_prompt}]},
          {"role": "user", "content": [{"type": "text", "text": prompt}]},
        ]

        inputs = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
            enable_thinking=False,
            return_dict=True
        ).to(model.device)

        torch.cuda.synchronize()
        start = time.perf_counter()

        with torch.no_grad():
          outputs = model.generate(
              **inputs,
              max_new_tokens=2048,
              temperature=0.0,
              do_sample=False,
              use_cache=True,
              pad_token_id=tokenizer.eos_token_id,
          )

        torch.cuda.synchronize()
        elapsed = time.perf_counter() - start

        input_len = inputs["input_ids"].shape[-1]
        generated_tokens = outputs[0][input_len:]
        resposta = tokenizer.decode(generated_tokens, skip_special_tokens=True)

      except Exception as e:
        resposta = f"ERROR: {e}\n{traceback.format_exc()}"
        elapsed = None

      writer.writerow({
              'prompt': prompt,
              'response': resposta,
              'question_id': questao.get("id"),
              'difficulty': questao.get("difficulty"),
              'time': elapsed,
          })

    if (i + 1) % 10 == 0:
      f.flush()
      os.fsync(f.fileno())
      shutil.copyfile('resultados.csv', drive_path)
      print(f"[{i+1}] processado — backup salvo no Drive")

shutil.copyfile('resultados.csv', drive_path)
print("Backup final salvo no Drive.")

## 8. Encerramento do ambiente de execução
Desconecta a sessão do Google Colabs para economizar recursos computacionais.

In [ ]:
from google.colab import runtime
runtime.unassign()